# Speech recognition with OWSM-CTC

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/espnet/notebook/blob/master/Demos/asr_demo.ipynb) [![asr_demo](https://github.com/espnet/notebook/actions/workflows/asr_demo.yml/badge.svg)](https://github.com/espnet/notebook/actions/workflows/asr_demo.yml)

One model transcribes speech in any of 150+ languages and tells you which
one it heard. No language flag needed, no search — one pass over the audio.

CPU is enough. The checkpoint is 4 GB, so the download is most of the wait.


## Install


In [ ]:
%pip install -q "espnet==202610.post1" espnet_model_zoo librosa


## A recording to work with

Any wav will do. This one is a sentence of read English.


In [ ]:
import librosa, soundfile as sf
from IPython.display import Audio, display

!wget -q -O sample.wav https://github.com/espnet/espnet/raw/master/test_utils/ctc_align_test.wav
speech, rate = librosa.load("sample.wav", sr=16000)
display(Audio(speech, rate=rate))


## Transcribe it

There are two OWSM v4 checkpoints, and this notebook runs on either: change
`MODEL` and nothing else.
[`owsm_ctc_v4_1B`](https://huggingface.co/espnet/owsm_ctc_v4_1B) is
encoder-only — `decode_long` reads a recording of any length, in overlapping
windows, and decodes each on the CTC head with no beam search — and
[`owsm_v4_medium_1B`](https://huggingface.co/espnet/owsm_v4_medium_1B) is the
encoder-decoder, which searches, is slower, and is the one that can be given
a text prompt.

In [ ]:
from espnet2.bin.s2t_inference import Speech2Text

MODEL = "espnet/owsm_ctc_v4_1B"  # or "espnet/owsm_v4_medium_1B"
s2t = Speech2Text.from_pretrained(MODEL, device="cpu")

segments = s2t.decode_long("sample.wav", lang_sym="<eng>", task_sym="<asr>")
print(" ".join(text for _, _, text in segments))

## Let it work out the language

`<nolang>` asks the model to identify the language instead of being told it.
The answer comes back as the first symbol of the decoded text.

`best_path` is CTC decoding with no search. On the encoder-only checkpoint
that is the whole model; on the encoder-decoder it reads the CTC *branch*,
which answers what that branch was trained on rather than what the task
symbol asks for — call `s2t(...)` there instead, and pay for the search.

In [ ]:
import librosa

window = librosa.util.fix_length(speech, size=16000 * 30)
text, *_ = s2t.best_path(window, lang_sym="<nolang>", task_sym="<asr>")[0]
print(text)


## Where next

- **From the terminal**: `pip install espnet && espnet asr sample.wav`
- **From the microphone**: `espnet asr --live` transcribes as you speak
- **In the browser**: the [owsm-ctc-v4 Space](https://huggingface.co/spaces/espnet/owsm-ctc-v4)
- **Translation** with the same model: [`st_demo.ipynb`](st_demo.ipynb)
